In [13]:
import math
import shapefile
import pandas as pd 
import numpy as np 

In [14]:
def read_shp(name_file):
    r = shapefile.Reader(name_file, encoding="utf-8")
    return [(rec.as_dict(), shp.points) for rec, shp in zip(r.records(), r.shapes())]

Here are the code lines to adapt an SHP file from an LHN to the standardized SHP format required by the program.

In [ ]:
tubos_shp=read_shp("Simulation_First_proposition_Réseau_+_Extension_Terme_Tube.shp")
nodes_shp=read_shp("Simulation_First_proposition_Réseau_+_Extension_Terme_Nœud.shp")
chaufferi_shp=read_shp("Simulation_First_proposition_Réseau_+_Extension_Terme_Centre_de_production.shp")
#Select the slack and the extra  plants 
chaufferi_shp[0][0]["ID"]="Extra_plant"
chaufferi_shp[1][0]["ID"]="Slack_plant"
# Change the id in the SHP original file in the pipe file
id_mapping = {
    "Chaufferie principale": "Extra_plant",
    "Chaufferie Belleroche": "Slack_plant"
}
for rec, points in tubos_shp:
    for campo in ["NodeUpsID", "NodeDwsID"]:
        if rec[campo] in id_mapping:
            rec[campo] = id_mapping[rec[campo]]
#Adding specifites about the fuel, nominal effciency, and emission factor, cost_constant 
chaufferi_shp[0][0]["Fuel_type"]=
chaufferi_shp[0][0]["Nominal_efficiency"]=
chaufferi_shp[0][0]["Emission_factor"]=
chaufferi_shp[0][0]["cost_constant"]=


# Update type of nodes 
for rec, points in nodes_shp:
    power = rec["Power"]
    if power == 0:
        rec["Type"] = "Node"
    elif power > 0:
        rec["Type"] = "Consumer"

In [22]:
chaufferi_shp[0][0]

{'Power': 23666173.8430294,
 'Flow': 146.408563977877,
 'VolFlowS': 0.149841029377417,
 'VolFlowR': 0.148941529820616,
 'Temperatur': 376.1470703125,
 'Temperatu0': 337.589709645706,
 'Temperatu1': 38.5602903542944,
 'PressureSu': 1228015.35203108,
 'PressureRe': 550000.0,
 'PressureCh': -678015.352031076,
 'PumpPower': 101288.682013333,
 'PumpCost': 0.0,
 'NoOfPumps': 1,
 'Production': 0.0,
 'ControlNod': 'NO_351',
 'DensitySup': 977.092252054537,
 'DensityRet': 982.993555620184,
 'HeadSupply': 290.759368583988,
 'HeadReturn': 219.743244564879,
 'ID': 'Extra_plant'}

Matrice d'incidence 

In [16]:
M_I_Zero=np.zeros((len(nodes_shp)+len(chaufferi_shp),len(tubos_shp)))
index=[]
columns=[]
for x in range(len(chaufferi_shp)):
    Name=chaufferi_shp[x][0]["ID"]
    index.append(Name)
for i in range(len(nodes_shp)):
    Name=nodes_shp[i][0]["ID"]
    index.append(Name)
for j in range (len(tubos_shp)):
    Name=tubos_shp[j][0]["ID"]
    columns.append(Name)
I_M=pd.DataFrame(M_I_Zero,index=index,columns=columns)
for i in range(len(tubos_shp)):
    Amont=tubos_shp[i][0]['NodeUpsID']
    Aval=tubos_shp[i][0]['NodeDwsID']
    Position_amont=I_M.index.get_loc(Amont)
    Position_aval=I_M.index.get_loc(Aval)
    I_M.iloc[Position_amont,i]=1
    I_M.iloc[Position_aval,i]=-1

Vecteur_Type

In [17]:
M_Type_Zero=np.zeros((1,len(nodes_shp)+len(chaufferi_shp)))
Index=["Type"]
Type=pd.DataFrame(M_Type_Zero,index=Index,columns=index,dtype=object)
#First Central production
for i in range(len(nodes_shp)+len(chaufferi_shp)):
    node_id=Type.columns[i]
    if node_id=="Slack_plant":
        Position_column=Type.columns.get_loc(node_id)
        Type.iloc[0, Position_column] ="Plant"
    elif node_id.startswith("Extra_plant"):
        Position_column=Type.columns.get_loc(node_id)
        Type.iloc[0, Position_column] ="Plant_support"
    else:
        Position_column=Type.columns.get_loc(node_id)
        Type.iloc[0, Position_column] =nodes_shp[i-len(chaufferi_shp)][0]["Type"] 

Matrice tuyau

In [18]:
M_pipe_Zero=np.zeros((4,len(tubos_shp)))
Index=["Length","inner_diameter","u_w_per_m2k","k_mm"]
Pipes=pd.DataFrame(M_pipe_Zero,index=Index,columns=columns,dtype=object)
#First Central production
for i in range(len(tubos_shp)):
    Length=tubos_shp[i][0]["Length"]/1000 # Result in km
    Diameter=tubos_shp[i][0]["DiameterSu"]*1000 # Result in mm 
    U_m_k=tubos_shp[i][0]["HeatCoeffS"]/(np.pi * tubos_shp[i][0]["DiameterSu"]) 
    k_mm=tubos_shp[i][0]["RoughnessS"]*1000  # Result in mm
    Pipes.iloc[0,i]=Length
    Pipes.iloc[1,i]=Diameter
    Pipes.iloc[2,i]=U_m_k
    Pipes.iloc[3,i]=k_mm

In [19]:
Pipes

,PI_20_1,PI_2073247.37291908,PI_2153929.43971579.1_1_1,PI_83_1,PI_2073334.57142939,PI_2115869.46194755.2,PI_2115869.46194755.1,PI_42_1,PI_42_2,PI_2166503.31799914.1_2,...,PI_2144498.63035481.1,PI_82_1,PI_82_2,PI_2074769.53629325,PI_2095547.92013338,PI_2085260.2012562.2,PI_2166503.31799914.2.1,PI_2166503.31799914.2.2,PI_2071661.97129973,PI_5
Length,0.005322,0.079932,0.012469,0.450821,0.031215,0.046071,0.141001,0.027426,0.12266,0.002632,...,0.229949,0.053293,0.026701,0.028418,0.115288,0.121977,0.294647,0.007562,0.02903,0.077092
inner_diameter,70.3,54.5,206.5,206.5,131.7,206.5,206.5,54.5,206.5,206.5,...,107.1,54.5,54.5,107.1,43.1,82.5,206.5,206.5,70.3,54.5
u_w_per_m2k,1.13197,1.226515,0.631995,0.631995,0.773418,0.631995,0.631995,1.226515,0.631995,0.631995,...,0.832183,1.226515,1.226515,0.832183,1.403222,1.003158,0.631995,0.631995,1.13197,1.226515
k_mm,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,...,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1


Energy Storage Matrix 

In [ ]:
# First verify how much nodes has storage tanks
N_storage=0
positions=[]
ids=[]
for i in range(len(nodes_shp)):
   if nodes_shp[i][0]["Type"]=="Storage":
      positions.append(i)
      ids.append(nodes_shp[i][0]["ID"])
      N_storage=+1
if N_storage>0:
    M_Storage_Zero=np.zeros((6,N_storage))
    Index=["Vtot","T_hot_init","T_cold_init","v_hot_fraction","UA_loss","UA_interface"]
    Storage=pd.DataFrame(M_Storage_Zero,index=Index,columns=ids,dtype=object)
    for i in  positions: 
        Vtot=nodes_shp[i][0]["Vtot"] # In m^3 
        T_hot_init=nodes_shp[i][0]["T_hot_init"] # In k 
        T_cold_init=nodes_shp[i][0]["T_cold_init"]# In k 
        v_hot_fraction=nodes_shp[i][0]["v_hot_fraction"]
        UA_loss=nodes_shp[i][0]["UA_loss"]
        UA_interface=nodes_shp[i][0]["UA_interface"]
        Storage.iloc[0,i]=Vtot
        Storage.iloc[1,i]=T_hot_init
        Storage.iloc[2,i]=T_cold_init
        Storage.iloc[3,i]=UA_loss
        Storage.iloc[4,i]=UA_interface
else:
    Storage = pd.DataFrame()


Central plant vector 

Heat pump matrix 

In [ ]:
# First verify how much nodes has storage tanks
N_Hp=0
positions=[]
ids=[]
for i in range(len(nodes_shp)):
   if nodes_shp[i][0]["Type"]=="Prosumer":
      positions.append(i)
      ids.append(nodes_shp[i][0]["ID"])
      N_Hp=+1
if N_Hp>0:
    M_HP_Zero=np.zeros((3,N_Hp))
    Index=["Vtot","T_hot_init"]
    Heat_pumps=pd.DataFrame(M_HP_Zero,index=Index,columns=ids,dtype=object)
    for i in  positions: 
        refrigerant =nodes_shp[i][0]["refrigerant"] # In m^3 
        dT_water=nodes_shp[i][0]["dT_water"] # In k 
        eta_s=nodes_shp[i][0]["eta_s"]# In k 
        Heat_pumps.iloc[0,i]=refrigerant
        Heat_pumps.iloc[1,i]=dT_water
        Heat_pumps.iloc[2,i]=eta_s
else:
    Heat_pumps = pd.DataFrame()

Multi_plant Matrix